In [ ]:
import json
import os
import random
import subprocess
import time
import traceback
from datetime import datetime
from functools import partialmethod
from io import StringIO
from multiprocessing import Pool
from pathlib import Path

import tqdm as tqdm_pbar
from colorama import Back, Fore, Style
from IPython.utils import io
from tqdm import tqdm, trange


In [1]:
nb_name = 3.9


In [ ]:
color_bars = [
    Fore.RED,
    Fore.GREEN,
    Fore.BLUE,
    Fore.MAGENTA,
    Fore.YELLOW,
    Fore.CYAN,
    Fore.WHITE,
]


In [ ]:
from medcat.cat import CAT

In [ ]:
import csv
import multiprocessing
import os

# import tqdm
import re
import sys
import warnings
from csv import writer
from multiprocessing import Pool

import numpy as np
import pandas as pd

# point to credentials and cogstack_v8_lite
sys.path.insert(0, "/data/AS/Samora/gloabl_files")
sys.path.insert(0, "/home/aliencat/samora/gloabl_files")
import pickle
from os.path import exists

from cogstack_v8_lite import *
from credentials import *
from scipy import stats

from COGStats import *

In [ ]:
def convert_date(date_string):
    date_string = date_string.split("T")[0]
    date_object = datetime.strptime(date_string, "%Y-%m-%d")
    return date_object


In [ ]:
def get_free_gpu():
    gpu_stats = subprocess.check_output(
        ["nvidia-smi", "--format=csv", "--query-gpu=memory.used,memory.free"]
    )
    gpu_df = pd.read_csv(
        StringIO(gpu_stats.decode("utf-8")),
        names=["memory.used", "memory.free"],
        skiprows=1,
    )
    print("GPU usage:\n{}".format(gpu_df))
    gpu_df["memory.free"] = gpu_df["memory.free"].map(lambda x: x.rstrip(" [MiB]"))
    idx = gpu_df["memory.free"].astype(int).idxmax()
    print(
        "Returning GPU{} with {} free MiB".format(idx, gpu_df.iloc[idx]["memory.free"])
    )
    return int(idx), gpu_df.iloc[idx]["memory.free"]


gpu_index, free_mem = get_free_gpu()

if int(free_mem) > 4000:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    print(f"Setting gpu with {free_mem} free")
else:
    print(f"Setting NO gpu, most free memory: {free_mem} !")
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


In [ ]:
import sys
import logging

nblog = open(f"{nb_name}.log", "w")
nblog = open(f"{nb_name}.log", "a+")
sys.stdout.echo = nblog
sys.stderr.echo = nblog

get_ipython().log.handlers[0].stream = nblog
get_ipython().log.setLevel(logging.INFO)

get_ipython().run_line_magic("autosave", "5")


In [ ]:
pwd

In [ ]:
treatment_doc_filename = "treatment_docs_v3.csv"
treatment_control_ratio_n = 1  # 1:n
pre_annotation_path = "current_pat_annots/"
pre_annotation_path_mrc = "current_pat_annots_mrc/"
store_annot = True


In [ ]:
from pathlib import Path

Path(pre_annotation_path).mkdir(parents=True, exist_ok=True)
Path(pre_annotation_path_mrc).mkdir(parents=True, exist_ok=True)


# Get treatment docs
# Examples:
#"adverse drug reaction" OR "adr" OR "Drug reaction" OR "Adverse reaction caused by drug" OR "Adverse reaction to medication" OR "Adverse reaction caused by drug" OR "Adverse drug effect" OR "adverse reaction" OR "anaphylaxis" OR "drug induced anaphylaxis" OR "Anaphylactic shock"
#0 docs African iron overload, Bantu siderosis
# 374 :  Transfusional haemosiderosis
#0 G320V
#1 Gly320Val
#0 Troisier-Hanot-Chauffard Syndrome
#4 Troisier
#0 Von Recklenhausen-Applebaum Disease
#0 Recklenhausen
#20 Iron Storage Disorder
#0 Pigmentary Cirrhosis
#C282Y/H63D
#22251 iron overload
#0 ser65-to-cys
#309 C282Y/H63D
#0 I105T
#0 G93R
#0 cys282-to-tyr
#221 CYS282TYR
#0 cys/tyr
#



In [ ]:
file_exists = exists(treatment_doc_filename)
if file_exists == False:
    docs = cohort_searcher_no_terms(
        index_name="epr_documents",
        fields_list="""client_idcode document_guid	document_description	body_analysed updatetime clientvisit_visitidcode""".split(),
        search_string=""" "Haemochromatosis" 
                        OR "Hemochromatosis" 
                        OR "HFE" 
                        OR "HHC" 
                        OR "c282y" 
                        OR "h63d" 
                        OR "S65C" 
                        OR "Cys282Tyr" 
                        OR "p.C282Y" 
                        OR "HHemochromatosis" 
                        OR "HLAH"
                        OR "Bronze diabetes"
                        OR "Bronzed cirrhosis"
                        OR "282y"
                        OR "282C/Y"
                        OR "rs1799945"
                        OR "rs1800562"
                        OR "rs1800730"
                        OR "c.187C>G"
                        OR "c.845G>A"
                        OR "c.193A>T"
                        OR "p.His63Asp"
                        OR "p. Cys282Tyr"
                        OR "p.Ser65Cys"
                        OR "Transfusional haemosiderosis"
                        OR "Gly320Val"
                        OR "Troisier"
                        OR "Iron Storage Disorder"
                        OR "C282Y/H63D"
                        
                         
                         
                        """,
    )
    docs.to_csv(treatment_doc_filename)
else:
    docs = pd.read_csv(treatment_doc_filename)


In [ ]:
len(docs["client_idcode"].unique())


In [ ]:
use_filter = False
if use_filter:
    json_filter_path = (
        "/data/AS/Samora/HFE/HFE/v18/MedCAT_Export_With_Text_2022-12-24_21_30_48.json"
    )

    with open(json_filter_path, "r") as f:
        json_data = json.load(f)

    len(json_data["projects"][0])
    json_cuis = json_data["projects"][0]["cuis"].split(",")
    cat.cdb.filter_by_cui(json_cuis)


In [ ]:
treatment_client_id_list = list(docs["client_idcode"].unique())


In [ ]:
len(treatment_client_id_list)

In [ ]:
random.seed(42)

use_controls = True
if use_controls:
    # Get control docs default 1:1

    all_idcodes = pd.read_csv("all_client_idcodes_epr_unique.csv")["client_idcode"]

    print(len(all_idcodes), len(treatment_client_id_list))

    full_control_client_id_list = list(set(all_idcodes) - set(treatment_client_id_list))

    full_control_client_id_list.sort()

    len(full_control_client_id_list) - len(all_idcodes)

    n_treatments = len(treatment_client_id_list) * treatment_control_ratio_n
    print(
        f"{n_treatments} selected as controls"
    )  # Soft control selection, many treatments will be false positives
    treatment_control_sample = pd.Series(full_control_client_id_list).sample(
        n_treatments, random_state=42
    )

    treatment_control_sample

    all_patient_list_control = list(treatment_control_sample.values)

    with open("control_list.pkl", "wb") as f:
        pickle.dump(all_patient_list_control, f)

    print(all_patient_list_control[0:10])


In [ ]:
pd.Series(full_control_client_id_list)

In [ ]:
all_patient_list = list(treatment_client_id_list)

In [ ]:
if use_controls:
    all_patient_list = all_patient_list + all_patient_list_control


In [ ]:
patient_concatted_master_list = []

In [ ]:
all_df_builder = None

In [ ]:
save_threshold = 200

failed_list = []

In [ ]:
current_pat_line_path = "current_pat_lines/"


Path(current_pat_line_path).mkdir(parents=True, exist_ok=True)


In [ ]:
random.shuffle(all_patient_list)

In [ ]:
print(len(all_patient_list))

In [ ]:
current_pat_line_path

In [ ]:
main_options = {
    "demo": True,
    "bmi": True,
    "bloods": True,
    "drugs": True,
    "diagnostics": True,
    "core_02": True,
    "bed": True,
    "vte_status": True,
    "hosp_site": True,
    "core_resus": True,
    "news": True,
    "annotations": True,
    "annotations_mrc": True,
}


In [ ]:
def get_demo(current_pat_client_id_code):
    current_pat_demo = get_demographics2([current_pat_client_id_code])

    current_pat_demo

    current_pat_demo = append_age_at_record_series(current_pat_demo)

    demo_dataframe = pd.DataFrame(current_pat_demo).T

    demo_dataframe.reset_index(inplace=True)

    current_pat_demo = EthnicityAbstractor.abstractEthnicity(
        demo_dataframe,
        outputNameString="_census",
        ethnicityColumnString="client_racecode",
    )

    dummied_dummy_ethnicity_dataframe = pd.DataFrame(
        data=[(np.zeros(5))],
        columns=[
            "census_white",
            "census_asian_or_asian_british",
            "census_black_african_caribbean_or_black_british",
            "census_mixed_or_multiple_ethnic_groups",
            "census_other_ethnic_group",
        ],
    )

    cen_res = pd.get_dummies(current_pat_demo["census"], prefix="census")

    dummied_dummy_ethnicity_dataframe[cen_res.columns[0]] = cen_res[cen_res.columns[0]]

    abstrated_ethnicity_dummied = dummied_dummy_ethnicity_dataframe

    current_pat_demo = pd.concat(
        [current_pat_demo, abstrated_ethnicity_dummied], axis=1
    )

    current_pat_demo.reset_index(inplace=True)

    current_pat_demo

    sex_map = {"Male": 1, "Female": 0, "male": 1, "female": 0}

    current_pat_demo["male"] = current_pat_demo["client_gendercode"].map(sex_map)

    current_pat_demo["dead"] = int(type(current_pat_demo["client_deceaseddtm"]) == str)

    current_pat_demo = current_pat_demo[
        [
            "client_idcode",
            "male",
            "age",
            "dead",
            "census_white",
            "census_asian_or_asian_british",
            "census_black_african_caribbean_or_black_british",
            "census_mixed_or_multiple_ethnic_groups",
            "census_other_ethnic_group",
        ]
    ].copy()

    current_pat_demo["dead"] = current_pat_demo["dead"].astype(int)

    current_pat_demo["age"] = current_pat_demo["age"].astype(int)

    current_pat_demo["dead"] = current_pat_demo["dead"].astype(float)

    current_pat_demo["age"] = current_pat_demo["age"].astype(float)

    current_pat_demo["male"] = current_pat_demo["male"].astype(float)

    return current_pat_demo


In [ ]:
def get_smoking(current_pat_client_id_code):
    search_term = "CORE_SmokingStatus"
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string=f'obscatalogmasteritem_displayname:("{search_term}") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    term = "smoking_status".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        value_array = features_data["observation_valuetext_analysed"].dropna()

        features[f"{term}_current"] = value_array.str.contains("Current Smoker").astype(
            int
        )
        features[f"{term}_non"] = value_array.str.contains("Non-Smoker").astype(int)

    else:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        features[f"{term}_current"] = np.nan
        features[f"{term}_non"] = np.nan

    return features

In [ ]:
def get_core_02(current_pat_client_id_code):
    search_term = "CORE_SpO2"
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string=f'obscatalogmasteritem_displayname:("{search_term}") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    term = "core_sp_02".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        all_terms = list(
            features_data["observation_valuetext_analysed"].dropna().unique()
        )

        for term in all_terms:
            features[f'{term.replace("-", "_").replace("%", "pct")}'] = 1
            # features[f'{bed_term}'] = 1

    return features

In [ ]:
def get_bed(current_pat_client_id_code):
    search_term = "CORE_BedNumber3"
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string=f'obscatalogmasteritem_displayname:("{search_term}") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    term = "bed".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        all_bed_terms = list(
            features_data["observation_valuetext_analysed"].dropna().unique()
        )

        for bed_term in all_bed_terms:
            features[f"bed_{bed_term}"] = 1

    return features

In [ ]:
def get_vte_status(current_pat_client_id_code):
    search_term = "CORE_VTE_STATUS"
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string=f'obscatalogmasteritem_displayname:("{search_term}") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    term = "VTE_Status".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        di = {
            "High risk of VTE High risk of bleeding": 1,
            "High risk of VTE Low risk of bleeding": 0,
        }

        value_array = features_data["observation_valuetext_analysed"].map(di)

        value_array = value_array.astype(float)

        features[f"{term}_mean"] = value_array.mean()
        features[f"{term}_median"] = value_array.median()
        features[f"{term}_std"] = value_array.std()
        features[f"{term}_max"] = max(value_array)
        features[f"{term}_min"] = min(value_array)
        features[f"{term}_n"] = value_array.shape[0]
    else:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        features[f"{term}_mean"] = np.nan
        features[f"{term}_median"] = np.nan
        features[f"{term}_std"] = np.nan
        features[f"{term}_max"] = np.nan
        features[f"{term}_min"] = np.nan
        features[f"{term}_n"] = np.nan

    return features

In [ ]:
def get_hosp_site(current_pat_client_id_code):
    search_term = "CORE_HospitalSite"
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string=f'obscatalogmasteritem_displayname:("{search_term}") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == search_term
    ].copy()

    term = "hosp_site".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        value_array = features_data["observation_valuetext_analysed"].dropna()

        features[f"{term}_dh"] = value_array.str.contains("DH").astype(int)
        features[f"{term}_ph"] = value_array.str.contains("PRUH").astype(int)

    else:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        features[f"{term}_dh"] = np.nan
        features[f"{term}_ph"] = np.nan

    return features

In [ ]:
def get_core_resus(current_pat_client_id_code):
    current_pat_raw = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='obscatalogmasteritem_displayname:("CORE_RESUS_STATUS") ',
    )

    if len(current_pat_raw) == 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == "CORE_RESUS_STATUS"
    ].copy()

    # screen and purge dud values

    features_data.dropna(inplace=True)

    # -----------------------------------------------------------------

    features_data = current_pat_raw[
        current_pat_raw["obscatalogmasteritem_displayname"] == "CORE_RESUS_STATUS"
    ].copy()

    features_data.dropna(inplace=True)

    term = "CORE_RESUS_STATUS".lower()

    if len(features_data) > 0:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        value_array = (
            features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        features[f"{term}_mean"] = value_array.mean()
        features[f"{term}_median"] = value_array.median()
        features[f"{term}_std"] = value_array.std()
        features[f"{term}_max"] = max(value_array)
        features[f"{term}_min"] = min(value_array)
        features[f"{term}_n"] = value_array.shape[0]
    else:
        features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        features[f"{term}_mean"] = np.nan
        features[f"{term}_median"] = np.nan
        features[f"{term}_std"] = np.nan
        features[f"{term}_max"] = np.nan
        features[f"{term}_min"] = np.nan
        features[f"{term}_n"] = np.nan

    return features

In [ ]:
def get_news(current_pat_client_id_code):
    current_pat_raw_news = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='obscatalogmasteritem_displayname:("NEWS" OR "NEWS2") ',
    )

    news_features = pd.DataFrame(
        data=[current_pat_client_id_code], columns=["client_idcode"]
    )

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS2_Score"
    ].copy()

    # screen and purge dud values
    news_features_data = news_features_data[
        (news_features_data["observation_valuetext_analysed"].astype(float) < 20)
        & (news_features_data["observation_valuetext_analysed"].astype(float) > -20)
    ].copy()

    if len(news_features_data) > 0:
        news_features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_score_mean"] = value_array.mean()
        news_features["news_score_median"] = value_array.median()
        news_features["news_score_std"] = value_array.std()
        news_features["news_score_max"] = max(value_array)
        news_features["news_score_min"] = min(value_array)
        news_features["news_score_n"] = value_array.shape[0]
    else:
        news_features["news_score_mean"] = np.nan
        news_features["news_score_median"] = np.nan
        news_features["news_score_std"] = np.nan
        news_features["news_score_max"] = np.nan
        news_features["news_score_min"] = np.nan
        news_features["news_score_n"] = np.nan

    # -----------------------------------------------------------------
    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_Systolic_BP"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"]
            .dropna()
            .dropna()
            .astype(float)
        )
        news_features["news_systolic_bp_mean"] = value_array.mean()
        news_features["news_systolic_bp_median"] = value_array.median()
        news_features["news_systolic_bp_std"] = value_array.std()
        news_features["news_systolic_bp_max"] = max(value_array)
        news_features["news_systolic_bp_min"] = min(value_array)
        news_features["news_systolic_bp_n"] = value_array.shape[0]
    else:
        news_features["news_systolic_bp_mean"] = np.nan
        news_features["news_systolic_bp_median"] = np.nan
        news_features["news_systolic_bp_std"] = np.nan
        news_features["news_systolic_bp_max"] = np.nan
        news_features["news_systolic_bp_min"] = np.nan
        news_features["news_systolic_bp_n"] = np.nan

    # -----------------------------------------------------------------
    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_Diastolic_BP"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_diastolic_bp_mean"] = value_array.mean()
        news_features["news_diastolic_bp_median"] = value_array.median()
        news_features["news_diastolic_bp_std"] = value_array.std()
        news_features["news_diastolic_bp_max"] = max(value_array)
        news_features["news_diastolic_bp_min"] = min(value_array)
        news_features["news_diastolic_bp_n"] = value_array.shape[0]
    else:
        news_features["news_diastolic_bp_mean"] = np.nan
        news_features["news_diastolic_bp_median"] = np.nan
        news_features["news_diastolic_bp_std"] = np.nan
        news_features["news_diastolic_bp_max"] = np.nan
        news_features["news_diastolic_bp_min"] = np.nan
        news_features["news_diastolic_bp_n"] = np.nan

    # -----------------------------------------------------------------
    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"]
        == "NEWS_Respiration_Rate"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_respiration_rate_mean"] = value_array.mean()
        news_features["news_respiration_rate_median"] = value_array.median()
        news_features["news_respiration_rate_std"] = value_array.std()
        news_features["news_respiration_rate_max"] = max(value_array)
        news_features["news_respiration_rate_min"] = min(value_array)
        news_features["news_respiration_rate_n"] = value_array.shape[0]
    else:
        news_features["news_respiration_rate_mean"] = np.nan
        news_features["news_respiration_rate_median"] = np.nan
        news_features["news_respiration_rate_std"] = np.nan
        news_features["news_respiration_rate_max"] = np.nan
        news_features["news_respiration_rate_min"] = np.nan
        news_features["news_respiration_rate_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_Heart_Rate"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_heart_rate_mean"] = value_array.mean()
        news_features["news_heart_rate_median"] = value_array.median()
        news_features["news_heart_rate_std"] = value_array.std()
        news_features["news_heart_rate_max"] = max(value_array)
        news_features["news_heart_rate_min"] = min(value_array)
        news_features["news_heart_rate_n"] = value_array.shape[0]
    else:
        news_features["news_heart_rate_mean"] = np.nan
        news_features["news_heart_rate_median"] = np.nan
        news_features["news_heart_rate_std"] = np.nan
        news_features["news_heart_rate_max"] = np.nan
        news_features["news_heart_rate_min"] = np.nan
        news_features["news_heart_rate_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"]
        == "NEWS_Oxygen_Saturation"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_oxygen_saturation_mean"] = value_array.mean()
        news_features["news_oxygen_saturation_median"] = value_array.median()
        news_features["news_oxygen_saturation_std"] = value_array.std()
        news_features["news_oxygen_saturation_max"] = max(value_array)
        news_features["news_oxygen_saturation_min"] = min(value_array)
        news_features["news_oxygen_saturation_n"] = value_array.shape[0]
    else:
        news_features["news_oxygen_saturation_mean"] = np.nan
        news_features["news_oxygen_saturation_median"] = np.nan
        news_features["news_oxygen_saturation_std"] = np.nan
        news_features["news_oxygen_saturation_max"] = np.nan
        news_features["news_oxygen_saturation_min"] = np.nan
        news_features["news_oxygen_saturation_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS Temperature"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_temperature_mean"] = value_array.mean()
        news_features["news_temperature_median"] = value_array.median()
        news_features["news_temperature_std"] = value_array.std()
        news_features["news_temperature_max"] = max(value_array)
        news_features["news_temperature_min"] = min(value_array)
        news_features["news_temperature_n"] = value_array.shape[0]
    else:
        news_features["news_temperature_mean"] = np.nan
        news_features["news_temperature_median"] = np.nan
        news_features["news_temperature_std"] = np.nan
        news_features["news_temperature_max"] = np.nan
        news_features["news_temperature_min"] = np.nan
        news_features["news_temperature_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_AVPU"
    ].copy()

    news_features_data.dropna(inplace=True)

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features["news_avpu_mean"] = value_array.mean()
        news_features["news_avpu_median"] = value_array.median()
        news_features["news_avpu_std"] = value_array.std()
        news_features["news_avpu_max"] = max(value_array)
        news_features["news_avpu_min"] = min(value_array)
        news_features["news_avpu_n"] = value_array.shape[0]
    else:
        news_features["news_avpu_mean"] = np.nan
        news_features["news_avpu_median"] = np.nan
        news_features["news_avpu_std"] = np.nan
        news_features["news_avpu_max"] = np.nan
        news_features["news_avpu_min"] = np.nan
        news_features["news_avpu_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"]
        == "NEWS_Supplemental_Oxygen"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "supplemental_oxygen"

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS2_Sp02_Target"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "Sp02_Target".lower()

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS2_Sp02_Scale"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "Sp02_Scale".lower()

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_Pulse_Type"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "pulse_type".lower()

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS_Pain_Score"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "Pain_Score".lower()

    if len(news_features_data) > 0:
        value_array = news_features_data["observation_valuetext_analysed"].astype(float)
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"] == "NEWS Oxygen Litres"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "oxygen_litres".lower()

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"]
        == "NEWS Oxygen Delivery"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "oxygen_delivery".lower()

    if len(news_features_data) > 0:
        value_array = (
            news_features_data["observation_valuetext_analysed"].dropna().astype(float)
        )
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    # -----------------------------------------------------------------

    news_features_data = current_pat_raw_news[
        current_pat_raw_news["obscatalogmasteritem_displayname"]
        == "NEWS Oxygen Delivery"
    ].copy()

    news_features_data.dropna(inplace=True)

    term = "oxygen_delivery".lower()

    if len(news_features_data) > 0:
        value_array = news_features_data["observation_valuetext_analysed"].astype(float)
        news_features[f"news_{term}_mean"] = value_array.mean()
        news_features[f"news_{term}_median"] = value_array.median()
        news_features[f"news_{term}_std"] = value_array.std()
        news_features[f"news_{term}_max"] = max(value_array)
        news_features[f"news_{term}_min"] = min(value_array)
        news_features[f"news_{term}_n"] = value_array.shape[0]
    else:
        news_features[f"news_{term}_mean"] = np.nan
        news_features[f"news_{term}_median"] = np.nan
        news_features[f"news_{term}_std"] = np.nan
        news_features[f"news_{term}_max"] = np.nan
        news_features[f"news_{term}_min"] = np.nan
        news_features[f"news_{term}_n"] = np.nan

    return news_features

In [ ]:
def get_current_pat_annotations_mrc_cs(current_pat_client_id_code):
    global start_time

    current_pat_docs = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='obscatalogmasteritem_displayname:("AoMRC_ClinicalSummary_FT") ',
    )

    n_docs_to_annotate = len(current_pat_docs)
    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations_mrc",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    annotation_map = {
        "True": 1,
        "Presence": 1,
        "Recent": 1,
        "Past": 0,
        "Subject/Experiencer": 1,
        "Other": 0,
        "Hypothetical": 0,
        "Patient": 1,
    }

    file_exists = exists(pre_annotation_path_mrc + current_pat_client_id_code)

    if file_exists == False:
        with io.capture_output() as captured:
            pats_anno_annotations = cat.get_entities_multi_texts(
                current_pat_docs["observation_valuetext_analysed"].dropna()
            )
            # , n_process=1
            if store_annot:
                with open(
                    pre_annotation_path_mrc + current_pat_client_id_code, "wb"
                ) as f:
                    pickle.dump(pats_anno_annotations, f)

    else:
        with open(pre_annotation_path + current_pat_client_id_code, "rb") as f:
            pats_anno_annotations = pickle.load(f)

    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations_mrc",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    sum_count = 0
    for i in range(0, len(pats_anno_annotations)):
        sum_count = sum_count + len(list(pats_anno_annotations[i]["entities"].keys()))

    sum_count_index_list = [x for x in range(0, sum_count)]
    all_doc_entities = {"entities": dict.fromkeys(sum_count_index_list, {})}
    sum_count_index = 0
    for i in range(0, len(pats_anno_annotations)):
        key_list = list(pats_anno_annotations[i]["entities"].keys())
        for j in range(0, len(key_list)):
            all_doc_entities["entities"][sum_count_index] = pats_anno_annotations[i][
                "entities"
            ].get(key_list[j])
            sum_count_index = sum_count_index + 1

    pats_anno_annotations = all_doc_entities

    all_cui_list = []

    all_meta_anno = False
    confidence_threshold_presence = 0.8
    confidence_threshold_subject = 0.8
    confidence_threshold_concept_accuracy = 0.8

    cui_list_pretty_names = []
    doc_keys = list(pats_anno_annotations.keys())
    for i in range(0, len(doc_keys)):
        current_pats_entry = pats_anno_annotations.get("entities")
        current_pats_entry_keys = list(pats_anno_annotations.get("entities").keys())
        for j in range(0, len(current_pats_entry_keys)):
            all_cui_list.append(
                current_pats_entry.get(current_pats_entry_keys[j])["cui"]
            )
            cui_list_pretty_names.append(
                current_pats_entry.get(current_pats_entry_keys[j])["pretty_name"]
            )

    cui_list = all_cui_list

    cui_list_pretty_names = list(set(cui_list_pretty_names))

    cui_list_pretty_names_meta_list = []
    if all_meta_anno:
        cui_list_pretty_names_meta = [x + "_meta" for x in cui_list_pretty_names]

        cui_list_pretty_names_meta_list = []

        meta_key_list = ["Time", "Presence", "Subject/Experiencer"]

        meta_sub_key_list = ["value"]  # ,'confidence', 'name']

        for i in range(0, len(cui_list_pretty_names)):
            for j in range(0, len(meta_key_list)):
                for k in range(0, len(meta_sub_key_list)):
                    cui_list_pretty_names_meta_list.append(
                        cui_list_pretty_names[i]
                        + "_"
                        + meta_key_list[j]
                        + "_"
                        + meta_sub_key_list[k]
                    )

        print(
            "len(set(cui_list_pretty_names_meta_list))",
            len(set(cui_list_pretty_names_meta_list)),
        )

    cui_list_pretty_names_count_list = []
    for i in range(0, len(cui_list_pretty_names)):
        cui_list_pretty_names_count_list.append(
            cui_list_pretty_names[i] + "_count_mrc_cs"
        )
        cui_list_pretty_names_count_list.append(
            cui_list_pretty_names[i] + "_count_subject_present_mrc_cs"
        )

    all_columns_to_append = cui_list_pretty_names_count_list
    if all_meta_anno:
        all_columns_to_append.append(cui_list_pretty_names_meta_list)

    dummy_data = np.empty((1, len(all_columns_to_append)))
    dummy_data[:] = np.nan
    df_pat_entry = pd.DataFrame(data=dummy_data, columns=all_columns_to_append)

    df_pat = df_pat_entry.copy()
    df_pat["client_idcode"] = current_pat_client_id_code
    df_pat["n_docs"] = len(current_pat_docs["observation_valuetext_analysed"])

    df_pat.reset_index(inplace=True)

    df_pat = df_pat[["client_idcode"]].copy()
    df_pat_target = df_pat.copy(deep=True)

    df_pat_target = df_pat_target.reindex(
        list(df_pat_target.columns) + all_columns_to_append, axis=1
    )

    a = list(df_pat_target.columns)
    b = cui_list_pretty_names_meta_list

    df_pat_target[[x for x in a if (x not in b)]] = df_pat_target[
        [x for x in a if (x not in b)]
    ].fillna(0)

    df_pat_target["client_idcode"].iloc[0] = current_pat_client_id_code

    df_pat_target = df_pat_target.copy()

    list_targ = [x for x in range(0, len(df_pat_target))]

    columns = list(cui_list_pretty_names_meta_list)

    df_pat_target = df_pat_target.copy()  # [0:1]

    entry_counter = 0
    meta_counter = 0
    i = 0

    annotations = pats_anno_annotations

    if annotations is not None:
        annotation_keys = list(annotations["entities"].keys())

        for j in range(0, len(annotation_keys)):
            cui = annotations["entities"][annotation_keys[j]].get("cui")
            if cui in cui_list:
                current_col_name = annotations["entities"][annotation_keys[j]].get(
                    "pretty_name"
                )
                current_col_meta = annotations["entities"][annotation_keys[j]].get(
                    "meta_anns"
                )

                df_pat_target.at[i, current_col_name + "_count_mrc_cs"] = (
                    df_pat_target.loc[i][current_col_name + "_count_mrc_cs"] + 1
                )

                if current_col_meta is not None:
                    if (
                        current_col_meta["Presence"]["value"] == "True"
                        and current_col_meta["Subject/Experiencer"]["value"]
                        == "Patient"
                        and current_col_meta["Presence"]["confidence"]
                        > confidence_threshold_presence
                        and current_col_meta["Subject/Experiencer"]["confidence"]
                        > confidence_threshold_presence
                        and annotations["entities"][annotation_keys[j]]["acc"]
                        > confidence_threshold_concept_accuracy
                    ):
                        df_pat_target.at[
                            i, current_col_name + "_count_subject_present_mrc_cs"
                        ] = (
                            df_pat_target.loc[i][
                                current_col_name + "_count_subject_present_mrc_cs"
                            ]
                            + 1
                        )

                    elif (
                        current_col_meta["Presence"]["value"] == "True"
                        and current_col_meta["Subject/Experiencer"]["value"]
                        == "Relative"
                        and current_col_meta["Presence"]["confidence"]
                        > confidence_threshold_presence
                        and current_col_meta["Subject/Experiencer"]["confidence"]
                        > confidence_threshold_presence
                        and annotations["entities"][annotation_keys[j]]["acc"]
                        > confidence_threshold_concept_accuracy
                    ):
                        if (
                            current_col_name + "_count_relative_present_mrc_cs"
                            in df_pat_target.columns
                        ):
                            df_pat_target.at[
                                i, current_col_name + "_count_relative_present_mrc_cs"
                            ] = (
                                df_pat_target.loc[i][
                                    current_col_name + "_count_relative_present_mrc_cs"
                                ]
                                + 1
                            )

                        else:
                            df_pat_target[
                                current_col_name + "_count_relative_present_mrc_cs"
                            ] = 1
                    else:
                        pass

                if all_meta_anno:
                    if len(list(current_col_meta.keys())) > 0:
                        for key in current_col_meta.keys():
                            sub_anot = current_col_meta.get(key)
                            if len(list(sub_anot.keys())) > 0:
                                for sub_key in sub_anot.keys():
                                    if sub_key in meta_sub_key_list:
                                        try:
                                            key_result = sub_anot.get(sub_key)
                                            if type(key_result) is str:
                                                key_result = annotation_map.get(
                                                    key_result
                                                )

                                            df_pat_target.at[
                                                i,
                                                current_col_name
                                                + "_"
                                                + str(key)
                                                + "_"
                                                + str(sub_key),
                                            ] = key_result

                                            meta_counter = meta_counter + 1
                                        except Exception as e:
                                            print(e)
                                            pass

                meta_counter = meta_counter + 1
                entry_counter = entry_counter + 1

            else:
                pass
    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations_mrc",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    return df_pat_target

In [ ]:
def get_bmi_features(current_pat_client_id_code):
    current_pat_raw_bmi = cohort_searcher_with_terms_and_search(
        index_name="observations",
        fields_list="""observation_guid client_idcode	obscatalogmasteritem_displayname	observation_valuetext_analysed	observationdocument_recordeddtm clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='obscatalogmasteritem_displayname:("OBS BMI" OR "OBS Weight" OR "OBS height")',
    )

    if (
        len(
            current_pat_raw_bmi[
                current_pat_raw_bmi["obscatalogmasteritem_displayname"]
                == "OBS BMI Calculation"
            ]
        )
    ) == 0:
        bmi_features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        )

    bmi_sample = current_pat_raw_bmi[
        current_pat_raw_bmi["obscatalogmasteritem_displayname"] == "OBS BMI Calculation"
    ]
    # screen and purge dud values
    bmi_sample = bmi_sample[
        (bmi_sample["observation_valuetext_analysed"].astype(float) < 200)
        & (bmi_sample["observation_valuetext_analysed"].astype(float) > 6)
    ]

    if len(bmi_sample) > 0:
        bmi_features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()

        value_array = bmi_sample["observation_valuetext_analysed"].astype(float)

        bmi_features["bmi_mean"] = value_array.mean()
        bmi_features["bmi_median"] = value_array.median()
        bmi_features["bmi_std"] = value_array.std()
        bmi_features["bmi_high"] = int(bool(value_array.median() > 24.9))
        bmi_features["bmi_low"] = int(bool(value_array.median() < 18.5))
        bmi_features["bmi_extreme"] = int(bool(value_array.median() > 30))
        bmi_features["bmi_max"] = max(value_array)
        bmi_features["bmi_min"] = min(value_array)

    else:
        bmi_features = pd.DataFrame(
            data=[current_pat_client_id_code], columns=["client_idcode"]
        ).copy()
        bmi_features["bmi_mean"] = np.nan
        bmi_features["bmi_median"] = np.nan
        bmi_features["bmi_std"] = np.nan
        bmi_features["bmi_high"] = np.nan
        bmi_features["bmi_low"] = np.nan
        bmi_features["bmi_extreme"] = np.nan
        bmi_features["bmi_max"] = np.nan
        bmi_features["bmi_min"] = np.nan

    height_sample = current_pat_raw_bmi[
        current_pat_raw_bmi["obscatalogmasteritem_displayname"] == "OBS Height"
    ]
    # screen and purge dud values
    height_sample = height_sample[
        (height_sample["observation_valuetext_analysed"].astype(float) < 300)
        & (height_sample["observation_valuetext_analysed"].astype(float) > 30)
    ]

    # get height features
    if len(height_sample) > 0:
        if len(current_pat_raw_bmi) > 0:
            value_array = height_sample["observation_valuetext_analysed"].astype(float)

            bmi_features["height_mean"] = value_array.mean()
            bmi_features["height_median"] = value_array.median()
            bmi_features["height_std"] = value_array.std()

    else:
        bmi_features["height_mean"] = np.nan
        bmi_features["height_median"] = np.nan
        bmi_features["height_std"] = np.nan

    weight_sample = current_pat_raw_bmi[
        current_pat_raw_bmi["obscatalogmasteritem_displayname"] == "OBS Weight"
    ]
    # screen and purge dud values
    weight_sample = weight_sample[
        (weight_sample["observation_valuetext_analysed"].astype(float) < 800)
        & (weight_sample["observation_valuetext_analysed"].astype(float) > 1)
    ]

    # get weight features
    if len(weight_sample) > 0:
        if len(current_pat_raw_bmi) > 0:
            value_array = weight_sample["observation_valuetext_analysed"].astype(float)

            bmi_features["weight_mean"] = value_array.mean()
            bmi_features["weight_median"] = value_array.median()
            bmi_features["weight_std"] = value_array.std()
            bmi_features["weight_max"] = max(value_array)
            bmi_features["weight_min"] = min(value_array)

    else:
        bmi_features["weight_mean"] = np.nan
        bmi_features["weight_median"] = np.nan
        bmi_features["weight_std"] = np.nan
        bmi_features["weight_max"] = np.nan
        bmi_features["weight_min"] = np.nan

    return bmi_features

In [ ]:
def get_current_pat_bloods(current_pat_client_id_code):
    current_pat_bloods = cohort_searcher_with_terms_and_search(
        index_name="basic_observations",
        fields_list=[
            "client_idcode",
            "basicobs_itemname_analysed",
            "basicobs_value_numeric",
            "basicobs_entered",
            "clientvisit_serviceguid",
        ],
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string="basicobs_value_numeric:*",
    )  # In kibana can we pull the mean and std of each blood test for a reference.

    current_pat_bloods["datetime"] = pd.Series(
        current_pat_bloods["basicobs_entered"]
    ).apply(convert_date)

    basicobs_itemname_analysed_list = list(
        current_pat_bloods["basicobs_itemname_analysed"].unique()
    )

    basicobs_itemname_analysed_df_dict = {
        elem: current_pat_bloods[current_pat_bloods.basicobs_itemname_analysed == elem]
        for elem in basicobs_itemname_analysed_list
    }

    df_unique = current_pat_bloods.copy()

    df_unique.drop_duplicates(subset="client_idcode", inplace=True)

    df_unique.reset_index(inplace=True)

    filtered_list = []

    obs_columns_list = basicobs_itemname_analysed_list

    obs_columns_set = list(set(obs_columns_list))

    filtered_column_list = filtered_list

    obs_columns_set_columns_for_df = []
    for i in range(0, len(obs_columns_set)):
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_mean")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_median")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_mode")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_std")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_num-tests")
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-since-last-test"
        )
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_max")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_min")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_most-recent")
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_earliest-test")
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-between-first-last"
        )
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_contains-extreme-low"
        )
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_contains-extreme-high"
        )

    orig_columns = list(df_unique.columns)

    comb_cols = orig_columns + obs_columns_set_columns_for_df

    df_unique = df_unique.reindex(comb_cols, axis=1)

    df_unique = df_unique.copy()
    df_unique.drop(
        [
            "index",
            "_index",
            "_id",
            "_score",
            "basicobs_itemname_analysed",
            "basicobs_value_numeric",
            "basicobs_entered",
            "clientvisit_serviceguid",
            "datetime",
        ],
        inplace=True,
        axis=1,
    )

    today = datetime.today()

    clients_id = current_pat_client_id_code

    df_unique_filtered = df_unique.copy()

    i = 0

    for j in range(0, len(basicobs_itemname_analysed_list)):
        col_name = basicobs_itemname_analysed_list[j]

        filtered_df = basicobs_itemname_analysed_df_dict.get(col_name)

        filtered_column_values = filtered_df.basicobs_value_numeric.astype(
            float
        )._get_numeric_data()

        df_len = len(filtered_df)

        if df_len >= 1:
            # Mean assurance*
            agg_val = float(filtered_column_values.values[0])

            df_unique_filtered.at[i, col_name + "_mean"] = agg_val

        if df_len >= 2:
            # try:
            # Mean
            agg_val = filtered_column_values.mean()

            df_unique_filtered.at[i, col_name + "_mean"] = agg_val

            # recent
            agg_val = filtered_df.sort_values(by="datetime").iloc[-1][
                "basicobs_value_numeric"
            ]

            df_unique_filtered.at[i, col_name + "_most-recent"] = agg_val

            # earliest-test
            agg_val = filtered_df.sort_values(by="datetime").iloc[0][
                "basicobs_value_numeric"
            ]
            df_unique_filtered.at[i, col_name + "_earliest-test"] = agg_val

            # days-since-last-test
            date_object = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = today - date_object

            agg_val = delta.days

            df_unique_filtered.at[i, col_name + "_days-since-last-test"] = agg_val

            # n tests

            agg_val = len(filtered_column_values)

            df_unique_filtered.at[i, col_name + "_num-tests"] = agg_val

        if df_len >= 3:
            # try:
            # median
            # agg_val = np.median(filtered_df['basicobs_value_numeric'].to_numpy())
            agg_val = filtered_column_values.median()
            df_unique_filtered.at[i, col_name + "_median"] = agg_val

            # mode
            # agg_val = stats.mode(filtered_df['basicobs_value_numeric'].to_numpy(), keepdims=True)[0][0]

            agg_val = stats.mode(filtered_column_values)[0][0]
            df_unique_filtered.at[i, col_name + "_mode"] = agg_val

            # std
            agg_val = filtered_column_values.std()
            df_unique_filtered.at[i, col_name + "_std"] = agg_val

            # min
            agg_val = min(filtered_column_values)
            df_unique_filtered.at[i, col_name + "_min"] = agg_val

            # max
            agg_val = max(filtered_column_values)
            df_unique_filtered.at[i, col_name + "_max"] = agg_val

            # contains extreme low
            col_name_mean = (
                basicobs_itemname_analysed_df_dict.get(col_name)
                .basicobs_value_numeric._get_numeric_data()
                .mean()
            )
            col_name_std = (
                basicobs_itemname_analysed_df_dict.get(col_name)
                .basicobs_value_numeric._get_numeric_data()
                .std()
            )

            col_name_low = col_name_mean - (col_name_std * 3)

            agg_val = int(float(min(filtered_column_values)) < col_name_low)
            df_unique_filtered.at[i, col_name + "_contains-extreme-low"] = agg_val

            # contains extreme high
            col_name_high = col_name_mean + (col_name_std * 3)

            agg_val = int(float(max(filtered_column_values)) > col_name_high)

            df_unique_filtered.at[i, col_name + "_contains-extreme-high"] = agg_val

            # days_between earliest and last

            earliest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            oldest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = earliest - oldest

            agg_val = delta.days

            df_unique_filtered.at[i, col_name + "_days-between-first-last"] = agg_val

            # current_pat_bloods = df_unique_filtered
    return df_unique_filtered

In [ ]:
def get_current_pat_drugs(current_pat_client_id_code):
    # Drugs
    drugs = cohort_searcher_with_terms_and_search(
        index_name="order",
        fields_list="""client_idcode order_guid	order_name	order_summaryline order_holdreasontext	order_entered clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='order_typecode:"medication"',
    )

    current_pat_diagnostics = drugs.copy()

    current_pat_diagnostics["datetime"] = (
        pd.Series(current_pat_diagnostics["order_entered"]).dropna().apply(convert_date)
    )

    order_name_list = list(current_pat_diagnostics["order_name"].unique())

    order_name_df_dict = {
        elem: current_pat_diagnostics[current_pat_diagnostics.order_name == elem]
        for elem in order_name_list
    }

    df_unique = current_pat_diagnostics.copy()

    df_unique.drop_duplicates(subset="client_idcode", inplace=True)

    df_unique.reset_index(inplace=True)

    filtered_list = []

    obs_columns_list = order_name_list

    obs_columns_set = list(set(obs_columns_list))

    filtered_column_list = filtered_list

    obs_columns_set_columns_for_df = []
    for i in range(0, len(obs_columns_set)):
        obs_columns_set_columns_for_df.append(obs_columns_set[i] + "_num-drug-order")
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-since-last-drug-order"
        )
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-between-first-last-drug"
        )

    orig_columns = list(df_unique.columns)

    comb_cols = orig_columns + obs_columns_set_columns_for_df

    df_unique = df_unique.reindex(comb_cols, axis=1)

    df_unique = df_unique.copy()
    df_unique.drop(
        [
            "_index",
            "_id",
            "_score",
            "order_guid",
            "order_name",
            "order_summaryline",
            "order_holdreasontext",
            "order_entered",
            "clientvisit_visitidcode",
        ],
        inplace=True,
        axis=1,
    )

    today = datetime.today()

    clients_id = current_pat_client_id_code

    df_unique_filtered = df_unique.copy()

    i = 0

    for j in range(0, len(obs_columns_list)):
        col_name = obs_columns_list[j]

        filtered_df = order_name_df_dict.get(col_name)

        df_len = len(filtered_df)
        if df_len >= 1:
            # n tests

            agg_val = len(filtered_df)

            df_unique_filtered.at[i, col_name + "_num-drug-order"] = agg_val

            # days-since-last-test
            date_object = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = today - date_object

            agg_val = delta.days

            df_unique_filtered.at[i, col_name + "_days-since-last-drug-order"] = agg_val

        if df_len >= 2:
            # days_between earliest and last

            earliest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            oldest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = earliest - oldest

            agg_val = delta.days

            df_unique_filtered.at[
                i, col_name + "_days-between-first-last-drug"
            ] = agg_val

    return df_unique_filtered


In [ ]:
def get_current_pat_diagnostics(current_pat_client_id_code):
    # Diagnostic tests
    diagnostics = cohort_searcher_with_terms_and_search(
        index_name="order",
        fields_list="""client_idcode order_guid	order_name	order_summaryline order_holdreasontext	order_entered clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
        search_string='order_typecode:"diagnostic"',
    )

    current_pat_diagnostics = diagnostics.copy()

    current_pat_diagnostics["datetime"] = (
        pd.Series(current_pat_diagnostics["order_entered"]).dropna().apply(convert_date)
    )

    order_name_list = list(current_pat_diagnostics["order_name"].unique())

    order_name_df_dict = {
        elem: current_pat_diagnostics[current_pat_diagnostics.order_name == elem]
        for elem in order_name_list
    }

    df_unique = current_pat_diagnostics.copy()

    df_unique.drop_duplicates(subset="client_idcode", inplace=True)

    df_unique.reset_index(inplace=True)

    filtered_list = []

    obs_columns_list = order_name_list

    obs_columns_set = list(set(obs_columns_list))

    filtered_column_list = filtered_list

    obs_columns_set_columns_for_df = []
    for i in range(0, len(obs_columns_set)):
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_num-diagnostic-order"
        )
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-since-last-diagnostic-order"
        )
        obs_columns_set_columns_for_df.append(
            obs_columns_set[i] + "_days-between-first-last-diagnostic"
        )

    orig_columns = list(df_unique.columns)

    comb_cols = orig_columns + obs_columns_set_columns_for_df

    df_unique = df_unique.reindex(comb_cols, axis=1)

    df_unique = df_unique.copy()
    df_unique.drop(
        [
            "_index",
            "_id",
            "_score",
            "order_guid",
            "order_name",
            "order_summaryline",
            "order_holdreasontext",
            "order_entered",
            "clientvisit_visitidcode",
        ],
        inplace=True,
        axis=1,
    )

    today = datetime.today()

    clients_id = current_pat_client_id_code

    df_unique_filtered = df_unique.copy()

    i = 0

    for j in range(0, len(obs_columns_list)):
        col_name = obs_columns_list[j]

        filtered_df = order_name_df_dict.get(col_name)

        df_len = len(filtered_df)
        if df_len >= 1:
            # n tests

            agg_val = len(filtered_df)

            df_unique_filtered.at[i, col_name + "_num-diagnostic-order"] = agg_val

            # days-since-last-test
            date_object = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = today - date_object

            agg_val = delta.days

            df_unique_filtered.at[
                i, col_name + "_days-since-last-diagnostic-order"
            ] = agg_val

        if df_len >= 2:
            # days_between earliest and last

            earliest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            oldest = filtered_df.sort_values(by="datetime").iloc[-1]["datetime"]

            delta = earliest - oldest

            agg_val = delta.days

            df_unique_filtered.at[
                i, col_name + "_days-between-first-last-diagnostic"
            ] = agg_val

    return df_unique_filtered


In [ ]:
def get_current_pat_annotations(current_pat_client_id_code):
    global start_time

    current_pat_docs = cohort_searcher_with_terms_no_search(
        index_name="epr_documents",
        fields_list="""client_idcode document_guid	document_description	body_analysed updatetime clientvisit_visitidcode""".split(),
        term_name="client_idcode.keyword",
        entered_list=[current_pat_client_id_code],
    )
    n_docs_to_annotate = len(current_pat_docs)
    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    annotation_map = {
        "True": 1,
        "Presence": 1,
        "Recent": 1,
        "Past": 0,
        "Subject/Experiencer": 1,
        "Other": 0,
        "Hypothetical": 0,
        "Patient": 1,
    }

    file_exists = exists(pre_annotation_path + current_pat_client_id_code)

    if file_exists == False:
        with io.capture_output() as captured:
            pats_anno_annotations = cat.get_entities_multi_texts(
                current_pat_docs["body_analysed"].dropna()
            )
            # , n_process=1
        if store_annot:
            with open(pre_annotation_path + current_pat_client_id_code, "wb") as f:
                pickle.dump(pats_anno_annotations, f)

    else:
        with open(pre_annotation_path + current_pat_client_id_code, "rb") as f:
            pats_anno_annotations = pickle.load(f)

    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    sum_count = 0
    for i in range(0, len(pats_anno_annotations)):
        sum_count = sum_count + len(list(pats_anno_annotations[i]["entities"].keys()))

    sum_count_index_list = [x for x in range(0, sum_count)]
    all_doc_entities = {"entities": dict.fromkeys(sum_count_index_list, {})}
    sum_count_index = 0
    for i in range(0, len(pats_anno_annotations)):
        key_list = list(pats_anno_annotations[i]["entities"].keys())
        for j in range(0, len(key_list)):
            all_doc_entities["entities"][sum_count_index] = pats_anno_annotations[i][
                "entities"
            ].get(key_list[j])
            sum_count_index = sum_count_index + 1

    pats_anno_annotations = all_doc_entities

    all_cui_list = []

    all_meta_anno = False
    confidence_threshold_presence = 0.8
    confidence_threshold_subject = 0.8
    confidence_threshold_concept_accuracy = 0.8

    cui_list_pretty_names = []
    doc_keys = list(pats_anno_annotations.keys())
    for i in range(0, len(doc_keys)):
        current_pats_entry = pats_anno_annotations.get("entities")
        current_pats_entry_keys = list(pats_anno_annotations.get("entities").keys())
        for j in range(0, len(current_pats_entry_keys)):
            all_cui_list.append(
                current_pats_entry.get(current_pats_entry_keys[j])["cui"]
            )
            cui_list_pretty_names.append(
                current_pats_entry.get(current_pats_entry_keys[j])["pretty_name"]
            )

    cui_list = all_cui_list

    cui_list_pretty_names = list(set(cui_list_pretty_names))

    cui_list_pretty_names_meta_list = []
    if all_meta_anno:
        cui_list_pretty_names_meta = [x + "_meta" for x in cui_list_pretty_names]

        cui_list_pretty_names_meta_list = []

        meta_key_list = ["Time", "Presence", "Subject/Experiencer"]

        meta_sub_key_list = ["value"]  # ,'confidence', 'name']

        for i in range(0, len(cui_list_pretty_names)):
            for j in range(0, len(meta_key_list)):
                for k in range(0, len(meta_sub_key_list)):
                    cui_list_pretty_names_meta_list.append(
                        cui_list_pretty_names[i]
                        + "_"
                        + meta_key_list[j]
                        + "_"
                        + meta_sub_key_list[k]
                    )

        print(
            "len(set(cui_list_pretty_names_meta_list))",
            len(set(cui_list_pretty_names_meta_list)),
        )

    cui_list_pretty_names_count_list = []
    for i in range(0, len(cui_list_pretty_names)):
        cui_list_pretty_names_count_list.append(cui_list_pretty_names[i] + "_count")
        cui_list_pretty_names_count_list.append(
            cui_list_pretty_names[i] + "_count_subject_present"
        )

    all_columns_to_append = cui_list_pretty_names_count_list
    if all_meta_anno:
        all_columns_to_append.append(cui_list_pretty_names_meta_list)

    dummy_data = np.empty((1, len(all_columns_to_append)))
    dummy_data[:] = np.nan
    df_pat_entry = pd.DataFrame(data=dummy_data, columns=all_columns_to_append)

    df_pat = df_pat_entry.copy()
    df_pat["client_idcode"] = current_pat_client_id_code
    df_pat["n_docs"] = len(current_pat_docs["body_analysed"])

    df_pat.reset_index(inplace=True)

    df_pat = df_pat[["client_idcode"]].copy()
    df_pat_target = df_pat.copy(deep=True)

    df_pat_target = df_pat_target.reindex(
        list(df_pat_target.columns) + all_columns_to_append, axis=1
    )

    a = list(df_pat_target.columns)
    b = cui_list_pretty_names_meta_list

    df_pat_target[[x for x in a if (x not in b)]] = df_pat_target[
        [x for x in a if (x not in b)]
    ].fillna(0)

    df_pat_target["client_idcode"].iloc[0] = current_pat_client_id_code

    df_pat_target = df_pat_target.copy()

    list_targ = [x for x in range(0, len(df_pat_target))]

    columns = list(cui_list_pretty_names_meta_list)

    df_pat_target = df_pat_target.copy()  # [0:1]

    entry_counter = 0
    meta_counter = 0
    i = 0

    annotations = pats_anno_annotations

    if annotations is not None:
        annotation_keys = list(annotations["entities"].keys())

        for j in range(0, len(annotation_keys)):
            cui = annotations["entities"][annotation_keys[j]].get("cui")
            if cui in cui_list:
                current_col_name = annotations["entities"][annotation_keys[j]].get(
                    "pretty_name"
                )
                current_col_meta = annotations["entities"][annotation_keys[j]].get(
                    "meta_anns"
                )

                df_pat_target.at[i, current_col_name + "_count"] = (
                    df_pat_target.loc[i][current_col_name + "_count"] + 1
                )

                if current_col_meta is not None:
                    if (
                        current_col_meta["Presence"]["value"] == "True"
                        and current_col_meta["Subject/Experiencer"]["value"]
                        == "Patient"
                        and current_col_meta["Presence"]["confidence"]
                        > confidence_threshold_presence
                        and current_col_meta["Subject/Experiencer"]["confidence"]
                        > confidence_threshold_presence
                        and annotations["entities"][annotation_keys[j]]["acc"]
                        > confidence_threshold_concept_accuracy
                    ):
                        df_pat_target.at[
                            i, current_col_name + "_count_subject_present"
                        ] = (
                            df_pat_target.loc[i][
                                current_col_name + "_count_subject_present"
                            ]
                            + 1
                        )

                    elif (
                        current_col_meta["Presence"]["value"] == "True"
                        and current_col_meta["Subject/Experiencer"]["value"]
                        == "Relative"
                        and current_col_meta["Presence"]["confidence"]
                        > confidence_threshold_presence
                        and current_col_meta["Subject/Experiencer"]["confidence"]
                        > confidence_threshold_presence
                        and annotations["entities"][annotation_keys[j]]["acc"]
                        > confidence_threshold_concept_accuracy
                    ):
                        if (
                            current_col_name + "_count_relative_present"
                            in df_pat_target.columns
                        ):
                            df_pat_target.at[
                                i, current_col_name + "_count_relative_present"
                            ] = (
                                df_pat_target.loc[i][
                                    current_col_name + "_count_relative_present"
                                ]
                                + 1
                            )

                        else:
                            df_pat_target[
                                current_col_name + "_count_relative_present"
                            ] = 1
                    else:
                        pass

                if all_meta_anno:
                    if len(list(current_col_meta.keys())) > 0:
                        for key in current_col_meta.keys():
                            sub_anot = current_col_meta.get(key)
                            if len(list(sub_anot.keys())) > 0:
                                for sub_key in sub_anot.keys():
                                    if sub_key in meta_sub_key_list:
                                        try:
                                            key_result = sub_anot.get(sub_key)
                                            if type(key_result) is str:
                                                key_result = annotation_map.get(
                                                    key_result
                                                )

                                            df_pat_target.at[
                                                i,
                                                current_col_name
                                                + "_"
                                                + str(key)
                                                + "_"
                                                + str(sub_key),
                                            ] = key_result

                                            meta_counter = meta_counter + 1
                                        except Exception as e:
                                            print(e)
                                            pass

                meta_counter = meta_counter + 1
                entry_counter = entry_counter + 1

            else:
                pass
    update_pbar(
        current_pat_client_id_code,
        start_time,
        5,
        "annotations",
        n_docs_to_annotate=n_docs_to_annotate,
    )

    return df_pat_target

In [ ]:
def update_pbar(
    current_pat_client_id_code, start_time, stage_int, stage_str, **n_docs_to_annotate
):
    global colour_val
    global t

    colour_val = Fore.GREEN + Style.BRIGHT + stage_str

    t.set_description(
        f"n_skipped: {skipped_counter} | {current_pat_client_id_code} | task: {colour_val} | {n_docs_to_annotate}"
    )
    if (time.time() - start_time) > slow_execution_threshold_low:
        t.colour = Fore.YELLOW
        colour_val = Fore.YELLOW + stage_str
        t.set_description(
            f"n_skipped: {skipped_counter} | {current_pat_client_id_code} | task: {colour_val} | {n_docs_to_annotate}"
        )

    elif (time.time() - start_time) > slow_execution_threshold_high:
        t.colour = Fore.RED + Style.BRIGHT
        colour_val = Fore.RED + Style.BRIGHT + stage_str
        t.set_description(
            f"n_skipped: {skipped_counter} | {current_pat_client_id_code} | task: {colour_val} | {n_docs_to_annotate}"
        )

    elif (time.time() - start_time) > slow_execution_threshold_extreme:
        t.colour = Fore.RED + Style.DIM
        colour_val = Fore.RED + Style.DIM + stage_str
        t.set_description(
            f"n_skipped: {skipped_counter} | {current_pat_client_id_code} | task: {colour_val} | {n_docs_to_annotate}"
        )

    else:
        t.colour = Fore.GREEN + Style.DIM
        colour_val = Fore.GREEN + Style.DIM + stage_str
        t.set_description(
            f"n_skipped: {skipped_counter} | {current_pat_client_id_code} | task: {colour_val} | {n_docs_to_annotate}"
        )

    t.refresh()

In [ ]:
stripped_list_start = [x.replace(".csv", "") for x in os.listdir(current_pat_line_path)]


In [ ]:
def main(current_pat_client_id_code):
    global skipped_counter
    global start_time
    start_time = time.time()

    done_list = []
    if current_pat_client_id_code not in stripped_list_start:
        stripped_list = [
            x.replace(".csv", "") for x in os.listdir(current_pat_line_path)
        ]

        if current_pat_client_id_code not in stripped_list:
            try:
                patient_vector = []

                update_pbar(current_pat_client_id_code, start_time, 0, "demo")

                if main_options.get("demo"):
                    current_pat_demo = get_demo(current_pat_client_id_code)
                    patient_vector.append(current_pat_demo)

                update_pbar(current_pat_client_id_code, start_time, 1, "bmi")

                if main_options.get("bmi"):
                    bmi_features = get_bmi_features(current_pat_client_id_code)
                    patient_vector.append(bmi_features)

                update_pbar(current_pat_client_id_code, start_time, 2, "bloods")

                if main_options.get("bloods"):
                    current_pat_bloods = get_current_pat_bloods(
                        current_pat_client_id_code
                    )
                    patient_vector.append(current_pat_bloods)

                update_pbar(current_pat_client_id_code, start_time, 3, "drugs")

                if main_options.get("drugs"):
                    current_pat_drugs = get_current_pat_drugs(
                        current_pat_client_id_code
                    )
                    patient_vector.append(current_pat_drugs)

                update_pbar(current_pat_client_id_code, start_time, 4, "diagnostics")

                if main_options.get("diagnostics"):
                    current_pat_diagnostics = get_current_pat_diagnostics(
                        current_pat_client_id_code
                    )
                    patient_vector.append(current_pat_diagnostics)

                if main_options.get("annotations"):
                    df_pat_target = get_current_pat_annotations(
                        current_pat_client_id_code
                    )
                    patient_vector.append(df_pat_target)

                if main_options.get("annotations_mrc"):
                    df_pat_target = get_current_pat_annotations_mrc_cs(
                        current_pat_client_id_code
                    )
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 1, "core_02")

                if main_options.get("core_02"):
                    df_pat_target = get_core_02(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 2, "bed")

                if main_options.get("bed"):
                    df_pat_target = get_bed(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 3, "vte_status")

                if main_options.get("vte_status"):
                    df_pat_target = get_vte_status(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 4, "hosp_site")

                if main_options.get("hosp_site"):
                    df_pat_target = get_hosp_site(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 1, "core_resus")

                if main_options.get("core_resus"):
                    df_pat_target = get_core_resus(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                update_pbar(current_pat_client_id_code, start_time, 2, "news")

                if main_options.get("news"):
                    df_pat_target = get_news(current_pat_client_id_code)
                    patient_vector.append(df_pat_target)

                pat_concatted = pd.concat(patient_vector, axis=1)

                pat_concatted.drop("client_idcode", axis=1, inplace=True)

                pat_concatted.insert(0, "client_idcode", current_pat_client_id_code)

                pat_concatted.to_csv(
                    current_pat_line_path + str(current_pat_client_id_code) + ".csv"
                )

            except RuntimeError as RuntimeError_exception:
                print(RuntimeError)
                print("sleeping 1h")
                time.sleep(3600)

            except Exception as e:
                print(e)
                template = "An exception of type {0} occurred. Arguments:\n{1!r}"
                message = template.format(type(e).__name__, e.args)
                print(message)

        else:
            skipped_counter = skipped_counter + 1
            pass

In [ ]:
slow_execution_threshold_low = 10
slow_execution_threshold_high = 30
slow_execution_threshold_extreme = 60


In [ ]:
stripped_list = [x.replace(".csv", "") for x in os.listdir(current_pat_line_path)]


In [ ]:
all_patient_list = [fruit for fruit in all_patient_list if fruit not in stripped_list]

In [ ]:
priority_list_bool = False

if priority_list_bool:
    df_old_done = pd.read_csv(
        "HFE/v18/current_pat_lines_parts/current_pat_lines__part_0_merged.csv",
        usecols=["client_idcode", "Hemochromatosis (disorder)_count_subject_present"],
    )

    priority_list = df_old_done[
        df_old_done["Hemochromatosis (disorder)_count_subject_present"] > 0
    ]["client_idcode"].to_list()

    all_patient_list = priority_list  # + all_patient_list


In [ ]:
cat = CAT.load_model_pack("/HFE/" + "medcat_models/medcat_model_pack_bec3865f4a29ee20")

In [ ]:
skipped_counter = 0
t = trange(
    len(all_patient_list),
    desc="Bar desc",
    leave=True,
    colour="GREEN",
    position=0,
    total=len(all_patient_list),
)


In [ ]:
multi_process = False
if multi_process == False:
    random.shuffle(all_patient_list)
    for i in t:
        try:
            main(all_patient_list[i])
        except Exception as e:
            print(e)
            print(all_patient_list[i])

else:
    t = trange(
        len(all_patient_list),
        desc="Bar desc",
        leave=True,
        colour="GREEN",
        position=0,
        total=len(all_patient_list),
    )
    skipped_counter = 0

    if __name__ == "__main__":
        pool = Pool(8)
        # pool = eventlet.GreenPool(size=1000)
        for patient in pool.imap(main, all_patient_list):
            random.shuffle(all_patient_list)

            _

In [ ]:
break